In [ ]:
# Using liftover annotations
import polars as pl

# Config variables
liftover = "rheMac10ToHg38"
dataset_subset = "WES3+WGS3_outer_join.rh_SPF_U42"
#dataset_subset = "U42_WES.all2"

dfs = []
for chrom in [str(num) for num in list(range(1, 21))]:
    file = f"/master/abagwell/variant-analysis/results/rhesus/liftover/{liftover}/annotated/{dataset_subset}.SNP.chr{chrom}.tsv"

    dfs.append(pl.read_csv(file, separator="\t", has_header=True, comment_prefix='##'))

df = pl.concat(dfs)

In [ ]:
ClinVar_CLNDN = df.filter(
    pl.col("Extra").str.contains("ClinVar_CLNDN")
).with_columns(
    ClinVar_CLNDN = pl.col("Extra").str.split(";").list.get(2).str.replace("ClinVar_CLNDN=", "").str.split("|")
).explode("ClinVar_CLNDN"
).group_by("ClinVar_CLNDN").agg(pl.len()).sort("len")

ClinVar_CLNDN

In [ ]:
ClinVar_CLNDN.write_csv(f"/master/abagwell/variant-analysis/results/rhesus/annotations/{dataset_subset}.SNP.ClinVar_CLNDN.tsv",
    separator='\t')

In [ ]:
ClinVar_CLNSIG = df.select("Extra").with_columns(
    pl.col("Extra").str.split(";")
).explode("Extra").with_columns(
    pl.col("Extra").str.split("=")
).with_columns(
    key = pl.col("Extra").list.get(0),
    value = pl.col("Extra").list.get(1),
).drop("Extra").sort("key").filter(
    pl.col("key") == "ClinVar_CLNSIG"
).group_by("value").len().sort("len").rename({"value": "ClinVar_CLNSIG", "len": "Count"})
ClinVar_CLNSIG


In [ ]:
ClinVar_CLNSIG.write_csv(f"/master/abagwell/variant-analysis/results/rhesus/annotations/{dataset_subset}.SNP.ClinVar_CLNSIG.tsv",
    separator='\t')